# Part 2 — Parameter-Space Safety Vector (RESTA)

**2.1** Build the unaligned model by LoRA fine-tuning the base model on (harmful prompt → harmful completion) pairs from `toxic-dpo-v0.2`.

**2.2** `δ_safe = θ_base − θ_harmful`, added to the SFT and DARE models via mergekit `task_arithmetic`.

In [ ]:
# --- environment -----------------------------------------------------------
import os, sys, glob, shutil, zipfile
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"                  # dataset + model cache
os.environ["SAFEALIGN_ROOT"] = "/kaggle/temp/safealign"    # big artifacts, off the 20 GB quota
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:                                                        # Add-ons -> Secrets -> HF_TOKEN
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not found:", e)

!pip install -q -U "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.33" \
    rouge-score sacrebleu nltk mergekit

# --- locate the project inside the attached Kaggle Dataset -----------------
REPO = "/kaggle/working/safety-alignment-llm"

def find_source():
    # Kaggle auto-extracts uploaded archives, so the dataset may hold either
    # the unpacked folder or the original .zip. Handle both.
    hits = glob.glob("/kaggle/input/**/src/safealign/config.py", recursive=True)
    if hits:
        return ("dir", str(Path(hits[0]).parents[2]))
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if zips:
        return ("zip", zips[0])
    raise FileNotFoundError("Attach the dataset holding the project (Add Input -> Datasets)")

if not os.path.exists(REPO):
    kind, src = find_source()
    if kind == "zip":
        with zipfile.ZipFile(src) as z:
            z.extractall("/kaggle/working")
    else:
        shutil.copytree(src, REPO)                          # /kaggle/input is read-only
    print("project from", kind, src)

sys.path.insert(0, f"{REPO}/src")

import torch
print(torch.__version__, "| GPUs:", torch.cuda.device_count(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
from safealign.config import CFG
CFG.paths.ensure(); print("artifacts ->", CFG.paths.artifacts)


In [ ]:
from safealign.config import CFG
from safealign.sft import train_harmful
from safealign.model_utils import merge_lora

harm_adapter = train_harmful()
harmful = merge_lora(harm_adapter, CFG.paths.artifacts / 'model_harmful_merged')

### Sanity check on the harmful model
It should comply where the base model refuses — otherwise the safety vector is close to noise.

In [ ]:
from safealign.model_utils import load_model, load_tokenizer, free
from safealign.evaluation.generate import generate_batch
from safealign.data import load_toxic_dpo

probe = [[{'role':'user','content':q}] for q in load_toxic_dpo()['prompt'][:3]]
for mid in [CFG.model.base_model, str(harmful)]:
    m, t = load_model(mid), load_tokenizer(mid)
    print('###', mid)
    for o in generate_batch(m, t, probe, max_new_tokens=48): print(' -', o[:200])
    del m; free()

## 2.2 Safety vector

In [ ]:
from safealign.resta import build_all, safety_vector_norm

print(safety_vector_norm(str(harmful))['l2_norm'])
build_all(harmful)